# Shared GRNO-2D for Navier--Stokes, SQG, and Vlasov--Poisson

This notebook is the paper-facing **rank-native 2D implementation**. It preserves
both the capacity and the direct centered--routed fusion of the original
Navier--Stokes/SQG GRNO, and applies the same trainable core to three tasks:

- 2D Kolmogorov Navier--Stokes in vorticity form;
- dissipative SQG;
- 1D1V Vlasov--Poisson, represented on the 2D phase-space grid `(v, x)`.

The trainable model is identical across all three tasks: same parameter names,
shapes, count, width, depths, route injection levels, decoder, and residual head.
Only a parameter-free task adapter changes:

\[
q_n \longrightarrow \text{carrier} \longrightarrow \text{departure map},
\]

plus boundary metadata and a fixed output projection. Separate weights are trained
for each PDE, as in standard neural-operator evaluation.

The notebook reuses the canonical datasets generated by the earlier experiment
notebooks. It does not silently generate a different benchmark.

## Public release: setup before execution

See [notebooks/README.md](README.md) for environment, configuration, dataset requirements, and output interpretation.

- The original code cells and hyperparameters are preserved; stored outputs and execution counts have been cleared.
- Configure `PROFILE`, `TASKS_TO_RUN`, `SEEDS`, and `DATA_ROOT_OVERRIDES` in the first code cell before executing subsequent sections in order. The shipped defaults are `paper`, all supported tasks, and seed `0`, with training and testing enabled.
- Canonical datasets, metadata, and normalization files are required. This notebook loads trajectories; it does not generate or download them. See [data/README.md](../data/README.md).
- `paper` and `pilot` require CUDA. `smoke` can run on CPU with valid data and uses a smaller model and shorter protocol.
- Check the printed `ROOT` and `device`. Set `ROOT` explicitly in the first code cell if automatic project discovery does not match your checkout.
- Each task trains its own model with a common architecture. These shared-core notebooks are an initial code release; the manuscript's experiment-specific protocols and baseline results require their corresponding experiments.

**Optional figure export:** the final nonempty code cell retains an absolute `PANEL_B_CACHE` path from the research machine. Edit it before execution or skip that cell. It requires the NS task, seed `0`, and a 40-step-compatible trained checkpoint and test trajectory.


## 1. Imports, paths, and frozen protocol

The shipped default trains all three tasks with seed `0` in the `paper` profile.
Select a single task for an initial setup check. Use `SEEDS = (0, 1, 2)` for three
independent seeds.


In [ ]:
from __future__ import annotations

import contextlib
import hashlib
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        if (parent / "grno_ns_sqg_data").is_dir() or (parent / "grno_mhd_vp_data").is_dir():
            return parent
        candidate = parent / "model" / "GRNO"
        if candidate.is_dir():
            return candidate.resolve()
    return cwd


ROOT = locate_project_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def short_hash(payload) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:12]


def atomic_torch_save(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".partial")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def safe_torch_load(path: Path, map_location=device):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(torch.as_tensor(state["torch_cpu"], dtype=torch.uint8, device="cpu"))
    if torch.cuda.is_available() and state.get("torch_cuda") is not None:
        torch.cuda.set_rng_state_all([
            torch.as_tensor(value, dtype=torch.uint8, device="cpu")
            for value in state["torch_cuda"]
        ])


def amp_context(enabled=True):
    if enabled and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()


PROFILE = "paper"                   # "smoke", "pilot", or "paper"
ALL_2D_TASKS = ("kolmogorov_ns", "dissipative_sqg", "vlasov_poisson")
TASKS_TO_RUN =ALL_2D_TASKS# ("dissipative_sqg",)  # final paper: ALL_2D_TASKS
SEEDS = (0,)                         # final paper: (0, 1, 2)
DATA_ROOT_OVERRIDES = {}
RUN_TRAINING = True
RUN_TESTING = True
RUN_ROUTE_INTERVENTIONS = True
RESUME_IF_AVAILABLE = True
MODEL_VERSION = "shared-grno2d-direct-route-v1"


@dataclass(frozen=True)
class RunConfig:
    profile: str
    epochs: int
    samples_per_epoch: int
    batch_size: int
    train_rollout: int
    val_rollout: int
    test_rollout: int
    width: int
    depths: tuple
    lr: float
    weight_decay: float
    grad_weight: float
    spectral_weight: float
    rollout_decay: float = 0.8
    use_amp: bool = True


if PROFILE == "paper":
    RUN = RunConfig(PROFILE, 100, 2048, 4, 4, 10, 40,
                    48, (2, 2, 3, 4), 3e-4, 1e-4, 0.05, 0.02)
elif PROFILE == "pilot":
    # Full paper architecture with a shorter optimization run.
    RUN = RunConfig(PROFILE, 20, 512, 4, 4, 10, 40,
                    48, (2, 2, 3, 4), 3e-4, 1e-4, 0.05, 0.02)
elif PROFILE == "smoke":
    RUN = RunConfig(PROFILE, 2, 16, 2, 2, 3, 5,
                    16, (1, 1, 1, 2), 1e-3, 1e-4, 0.03, 0.0,
                    use_amp=False)
else:
    raise ValueError(PROFILE)

if PROFILE != "smoke" and device.type != "cuda":
    raise RuntimeError("Pilot/paper mode requires CUDA; use PROFILE='smoke' on CPU.")
if not set(TASKS_TO_RUN).issubset(ALL_2D_TASKS):
    raise ValueError(TASKS_TO_RUN)

OUTPUT_ROOT = ROOT / "outputs" / "shared_grno2d" / PROFILE / MODEL_VERSION
MANIFEST_ROOT = ROOT / "manifests" / "shared_grno2d_K4_v1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "root": str(ROOT), "device": str(device), "profile": PROFILE,
    "tasks": TASKS_TO_RUN, "seeds": SEEDS, "run": asdict(RUN),
}, indent=2))

## 2. Task specifications and canonical datasets

The task specification contains equation metadata only. It does not change the
trainable architecture. Tensor order is `[channel, y, x]`; for Vlasov--Poisson,
`y` is the velocity coordinate and `x` is physical position.

In [ ]:
@dataclass(frozen=True)
class TaskSpec2D:
    name: str
    frame_dt: float
    lengths_yx: tuple
    boundary_yx: tuple
    route_kind: str
    condition_kind: str
    green_order: float = 0.0
    vmax: float = 0.0
    conserve_mean: bool = True


TASK_SPECS = {
    "kolmogorov_ns": TaskSpec2D(
        "kolmogorov_ns", 0.1, (2 * math.pi, 2 * math.pi),
        ("periodic", "periodic"), "fractional_active", "forcing",
        green_order=2.0),
    "dissipative_sqg": TaskSpec2D(
        "dissipative_sqg", 0.1, (2 * math.pi, 2 * math.pi),
        ("periodic", "periodic"), "fractional_active", "forcing",
        green_order=1.0),
    "vlasov_poisson": TaskSpec2D(
        "vlasov_poisson", 0.1, (12.0, 4 * math.pi),
        ("outflow", "periodic"), "vlasov_phase", "zero",
        vmax=6.0),
}


def _single_directory(candidates, description):
    candidates = sorted({Path(path).resolve() for path in candidates if Path(path).is_dir()})
    if not candidates:
        raise FileNotFoundError(
            f"No canonical dataset found for {description}. Run the original data-generation notebook first.")
    if len(candidates) > 1:
        raise RuntimeError(
            f"Ambiguous datasets for {description}; set DATA_ROOT_OVERRIDES explicitly:\n" +
            "\n".join(map(str, candidates)))
    return candidates[0]


def discover_data_root(task_name: str) -> Path:
    if task_name in DATA_ROOT_OVERRIDES:
        path = Path(DATA_ROOT_OVERRIDES[task_name]).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    if task_name in ("kolmogorov_ns", "dissipative_sqg"):
        direct = list((ROOT / "grno_ns_sqg_data").glob(f"{task_name}_paper_*"))
        recursive = list(ROOT.glob(f"**/grno_ns_sqg_data/{task_name}_paper_*"))
        return _single_directory([*direct, *recursive], task_name)
    if task_name == "vlasov_poisson":
        direct = list((ROOT / "grno_mhd_vp_data").glob("vlasov_poisson_paper_*"))
        recursive = list(ROOT.glob("**/grno_mhd_vp_data/vlasov_poisson_paper_*"))
        return _single_directory([*direct, *recursive], task_name)
    raise KeyError(task_name)


DATA_ROOTS = {name: discover_data_root(name) for name in TASKS_TO_RUN}


class TrajectoryStore2D:
    def __init__(self, task_name, split):
        self.task_name = task_name
        self.spec = TASK_SPECS[task_name]
        self.split = split
        self.root = DATA_ROOTS[task_name]
        self.regime = None
        if task_name in ("kolmogorov_ns", "dissipative_sqg"):
            self.states = np.load(self.root / f"{split}_states.npy", mmap_mode="r")
            self.conditions = np.load(self.root / f"{split}_forcing.npy", mmap_mode="r")
        else:
            self.states = np.load(self.root / f"{split}_states.npy", mmap_mode="r")
            self.conditions = None
            regime_path = self.root / f"{split}_regime.npy"
            if regime_path.exists():
                self.regime = np.load(regime_path, mmap_mode="r")
        self.n_trajectories = int(self.states.shape[0])
        self.n_frames = int(self.states.shape[1])

    def read_frame(self, trajectory, frame):
        value = np.asarray(self.states[int(trajectory), int(frame)], dtype=np.float32)
        if value.ndim == 2:
            value = value[None]
        if value.shape[0] != 1:
            raise ValueError((self.task_name, value.shape))
        return value

    def read_window(self, trajectory, start, horizon):
        values = np.asarray(
            self.states[int(trajectory), int(start):int(start) + int(horizon) + 1],
            dtype=np.float32)
        if values.ndim == 3:
            values = values[:, None]
        if values.shape[1] != 1:
            raise ValueError((self.task_name, values.shape))
        return values

    def read_condition(self, trajectory):
        shape = self.read_frame(trajectory, 0).shape[-2:]
        if self.conditions is None:
            return np.zeros((1, *shape), dtype=np.float32)
        value = np.asarray(self.conditions[int(trajectory)], dtype=np.float32)
        return value[None]

    def regime_label(self, trajectory):
        if self.regime is None:
            return None
        value = int(self.regime[int(trajectory)])
        return ("landau", "two_stream")[value]

    @property
    def max_train_start(self):
        return self.n_frames - RUN.train_rollout - 1

    def schema(self):
        return {
            "task": self.task_name, "split": self.split,
            "shape": list(self.states.shape), "dtype": str(self.states.dtype),
            "root": str(self.root),
        }


STORES = {
    name: {split: TrajectoryStore2D(name, split) for split in ("train", "val", "test")}
    for name in TASKS_TO_RUN
}


def validate_data_contract(task_name):
    root = DATA_ROOTS[task_name]
    manifest_path = root / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(manifest_path)
    manifest = json.loads(manifest_path.read_text())
    pde = manifest.get("pde", {})
    if task_name in ("kolmogorov_ns", "dissipative_sqg"):
        if manifest.get("dataset_version") != "active_scalar_etdrk2_v3":
            raise RuntimeError((task_name, manifest.get("dataset_version")))
        expected = {"n": 128, "domain_length": 2 * math.pi, "frame_dt": 0.1}
        if task_name == "kolmogorov_ns":
            expected.update({"green_kind": "poisson", "diffusivity": 1e-2, "substeps": 20})
        else:
            expected.update({"green_kind": "sqg", "diffusivity": 8e-3, "substeps": 40})
    else:
        if manifest.get("dataset_version") != "grno_mhd_vp_custom_pilot_v2":
            raise RuntimeError((task_name, manifest.get("dataset_version")))
        expected = {"nx": 128, "nv": 128, "Lx": 4 * math.pi,
                    "vmax": 6.0, "frame_dt": 0.1, "substeps": 5}
    for key, value in expected.items():
        actual = pde.get(key)
        if isinstance(value, float):
            if not math.isclose(float(actual), value, rel_tol=1e-9, abs_tol=1e-12):
                raise RuntimeError(f"{task_name}.{key}: expected {value}, found {actual}")
        elif actual != value:
            raise RuntimeError(f"{task_name}.{key}: expected {value!r}, found {actual!r}")
    return short_hash(manifest)


DATA_CONTRACT_HASHES = {name: validate_data_contract(name) for name in TASKS_TO_RUN}
for name in TASKS_TO_RUN:
    print(name, DATA_ROOTS[name])
    for split in ("train", "val", "test"):
        print(" ", STORES[name][split].schema())

## 3. Frozen preprocessing statistics

NS and SQG reuse the normalization files produced by the original GRNO dataset
notebook. VP uses a deterministic training-data estimator for state, delta, and
electric-field RMS scales. These are fixed preprocessing values, not trainable
modules.


In [ ]:
def estimate_vp_stats(max_pairs=512):
    store = STORES["vlasov_poisson"]["train"]
    spec = TASK_SPECS["vlasov_poisson"]
    rng = np.random.default_rng(17)
    values, deltas, electric_values = [], [], []
    dv = spec.lengths_yx[0] / store.states.shape[-2]
    nx = store.states.shape[-1]
    kx = 2 * math.pi * np.fft.fftfreq(nx, d=spec.lengths_yx[1] / nx)
    active = kx != 0
    if nx % 2 == 0:
        active[nx // 2] = False
    for _ in range(min(max_pairs, store.n_trajectories * (store.n_frames - 1))):
        trajectory = int(rng.integers(store.n_trajectories))
        frame = int(rng.integers(store.n_frames - 1))
        x = store.read_frame(trajectory, frame)[0]
        y = store.read_frame(trajectory, frame + 1)[0]
        values.append(x)
        deltas.append(y - x)
        rho = x.sum(axis=-2) * dv
        rh = np.fft.fft(rho)
        eh = np.zeros_like(rh)
        eh[active] = 1j * rh[active] / kx[active]
        electric_values.append(np.fft.ifft(eh).real)
    values = np.stack(values)
    deltas = np.stack(deltas)
    electric_values = np.stack(electric_values)
    return {
        "state_std": float(np.sqrt(np.mean(values ** 2)).clip(1e-6)),
        "delta_std": float(np.sqrt(np.mean(deltas ** 2)).clip(1e-7)),
        "carrier_std": [float(spec.vmax),
                        float(np.sqrt(np.mean(electric_values ** 2)).clip(1e-6))],
        "condition_std": 1.0,
        "source": "deterministic VP estimator, seed=17",
    }


def load_task_stats(task_name):
    root = DATA_ROOTS[task_name]
    if task_name in ("kolmogorov_ns", "dissipative_sqg"):
        path = root / "normalization.json"
        if not path.exists():
            raise FileNotFoundError(path)
        raw = json.loads(path.read_text())
        required = ("state_std", "velocity_std", "forcing_std", "delta_std")
        if not all(key in raw for key in required):
            raise RuntimeError((path, raw.keys()))
        return {
            "state_std": float(raw["state_std"]),
            "delta_std": float(raw["delta_std"]),
            "carrier_std": [float(raw["velocity_std"]), float(raw["velocity_std"])],
            "condition_std": float(raw["forcing_std"]),
            "source": str(path),
        }
    if task_name == "vlasov_poisson":
        return estimate_vp_stats()
    raise KeyError(task_name)


STATS = {name: load_task_stats(name) for name in TASKS_TO_RUN}
for name, stats in STATS.items():
    print(name, json.dumps(stats, indent=2))

## 4. Parameter-free carriers, traces, and boundary-aware sampling

All displacements are stored in physical units with channel order `(dx, dy)`. For
VP, `dy` means displacement along the velocity coordinate. The normal model uses
`route_mode='correct'`; other modes are same-weight mechanism interventions.

In [ ]:
_GRID_CACHE_2D = {}
_FOURIER_CACHE_2D = {}


def index_grid2d(h, w, device):
    key = (h, w, str(device))
    if key not in _GRID_CACHE_2D:
        _GRID_CACHE_2D[key] = torch.meshgrid(
            torch.arange(h, device=device, dtype=torch.float32),
            torch.arange(w, device=device, dtype=torch.float32),
            indexing="ij")
    return _GRID_CACHE_2D[key]


def _indices_and_valid(index, size, boundary):
    if boundary == "periodic":
        return torch.remainder(index, size), torch.ones_like(index, dtype=torch.bool)
    valid = (index >= 0) & (index < size)
    safe = index.clamp(0, size - 1)
    if boundary == "bounded":
        valid = torch.ones_like(valid)
    elif boundary != "outflow":
        raise ValueError(boundary)
    return safe, valid


def boundary_bilinear_sample(field, x_index, y_index, boundary_yx):
    """Sample BCHW at floating indices; y/x boundaries are handled independently."""
    b, c, h, w = field.shape
    by, bx = boundary_yx
    x = x_index.float()
    y = y_index.float()
    x0f, y0f = torch.floor(x), torch.floor(y)
    x0_raw, y0_raw = x0f.long(), y0f.long()
    x1_raw, y1_raw = x0_raw + 1, y0_raw + 1
    wx, wy = x - x0f, y - y0f
    flat = field.float().reshape(b, c, h * w)

    def take(yy_raw, xx_raw):
        xx, xvalid = _indices_and_valid(xx_raw, w, bx)
        yy, yvalid = _indices_and_valid(yy_raw, h, by)
        index = (yy * w + xx).reshape(b, 1, h * w).expand(-1, c, -1)
        value = torch.gather(flat, 2, index).reshape(b, c, h, w)
        return value * (xvalid & yvalid)[:, None]

    out = ((1 - wx) * (1 - wy))[:, None] * take(y0_raw, x0_raw)
    out += (wx * (1 - wy))[:, None] * take(y0_raw, x1_raw)
    out += ((1 - wx) * wy)[:, None] * take(y1_raw, x0_raw)
    out += (wx * wy)[:, None] * take(y1_raw, x1_raw)
    return out.to(field.dtype)


def boundary_warp2d(field, displacement, lengths_yx, boundary_yx):
    if field.ndim != 4 or displacement.shape != (field.shape[0], 2, *field.shape[-2:]):
        raise ValueError((field.shape, displacement.shape))
    ly, lx = map(float, lengths_yx)
    b, _, h, w = field.shape
    yy, xx = index_grid2d(h, w, field.device)
    xq = xx[None] + displacement[:, 0].float() / (lx / w)
    yq = yy[None] + displacement[:, 1].float() / (ly / h)
    return boundary_bilinear_sample(field, xq, yq, boundary_yx)


def fourier_grid_2d(h, w, ly, lx, device, rfft=False):
    key = (h, w, ly, lx, str(device), bool(rfft))
    if key not in _FOURIER_CACHE_2D:
        ky = 2 * math.pi * torch.fft.fftfreq(h, d=ly / h, device=device)[:, None]
        if rfft:
            kx = 2 * math.pi * torch.fft.rfftfreq(w, d=lx / w, device=device)[None, :]
        else:
            kx = 2 * math.pi * torch.fft.fftfreq(w, d=lx / w, device=device)[None, :]
        _FOURIER_CACHE_2D[key] = (ky, kx)
    return _FOURIER_CACHE_2D[key]


def fractional_active_carrier(state, spec):
    q = state[:, 0].float()
    h, w = q.shape[-2:]
    ly, lx = spec.lengths_yx
    ky, kx = fourier_grid_2d(h, w, ly, lx, q.device, rfft=True)
    k2 = kx.square() + ky.square()
    multiplier = torch.where(
        k2 > 0, -k2.pow(-0.5 * spec.green_order), torch.zeros_like(k2))
    qh = torch.fft.rfft2(q)
    ph = qh * multiplier
    ux = torch.fft.irfft2(1j * ky * ph, s=(h, w))
    uy = torch.fft.irfft2(-1j * kx * ph, s=(h, w))
    return torch.stack((ux, uy), dim=1)


def vp_carrier(state, spec):
    f = state[:, 0].float()
    b, nv, nx = f.shape
    dv = spec.lengths_yx[0] / nv
    rho = f.sum(-2) * dv
    kx = 2 * math.pi * torch.fft.fftfreq(nx, d=spec.lengths_yx[1] / nx,
                                         device=f.device)
    rh = torch.fft.fft(rho, dim=-1)
    eh = torch.zeros_like(rh)
    active = kx != 0
    if nx % 2 == 0:
        active[nx // 2] = False
    eh[:, active] = 1j * rh[:, active] / kx[active]
    electric = torch.fft.ifft(eh, dim=-1).real
    v = (-spec.vmax + torch.arange(nv, device=f.device) * (2 * spec.vmax / nv))
    v = v[None, :, None].expand(b, -1, nx)
    emap = electric[:, None, :].expand(-1, nv, -1)
    # Characteristic velocity in (x,v) is (v,-E).
    return torch.stack((v, -emap), dim=1)


def raw_carrier(state, spec):
    if spec.route_kind == "fractional_active":
        return fractional_active_carrier(state, spec)
    if spec.route_kind == "vlasov_phase":
        return vp_carrier(state, spec)
    raise ValueError(spec.route_kind)


def departure_from_carrier(carrier, spec, dt):
    dt = float(dt)
    if spec.route_kind == "vlasov_phase":
        v = carrier[:, 0]
        dx_half = -0.5 * dt * v
        half = torch.zeros_like(carrier)
        half[:, 0] = dx_half
        shifted = boundary_warp2d(carrier, half, spec.lengths_yx, spec.boundary_yx)
        electric_mid = -shifted[:, 1]
        dv = dt * electric_mid
        departed_v = v + dv
        dx = dx_half - 0.5 * dt * departed_v
        return torch.stack((dx, dv), dim=1)
    midpoint = boundary_warp2d(
        carrier, -0.5 * dt * carrier, spec.lengths_yx, spec.boundary_yx)
    return -dt * midpoint


def intervene_route(departure, mode):
    if mode == "correct":
        return departure
    if mode == "zero":
        return torch.zeros_like(departure)
    if mode == "reverse":
        return -departure
    if mode == "shuffle":
        return torch.roll(
            departure,
            shifts=(max(departure.shape[-2] // 3, 1),
                    max(departure.shape[-1] // 5, 1)),
            dims=(-2, -1))
    raise ValueError(mode)


def sampler_preflight():
    n, length = 64, 2 * math.pi
    yy, xx = index_grid2d(n, n, device)
    field = (torch.sin(2 * math.pi * xx / n) + 0.3 * torch.cos(4 * math.pi * yy / n))[None, None]
    departure = torch.zeros(1, 2, n, n, device=device)
    departure[:, 0] = 5 * length / n
    periodic = boundary_warp2d(field, departure, (length, length),
                               ("periodic", "periodic"))
    periodic_error = float((periodic - torch.roll(field, -5, -1)).abs().max())
    identity = boundary_warp2d(field, torch.zeros_like(departure),
                               (2.0, length), ("outflow", "periodic"))
    identity_error = float((identity - field).abs().max())
    print({"periodic_integer_shift_max": periodic_error,
           "mixed_identity_max": identity_error})
    assert periodic_error < 1e-5 and identity_error < 1e-5


sampler_preflight()

## 5. The one shared trainable model: `GRNO2DCore`

This is the original high-capacity NS/SQG encoder--decoder: native Conv2D,
width 48 in paper mode, four scales, and direct concatenation of centered and routed
features. Boundary rules are arguments to the forward pass and contain no weights.

In [ ]:
def pad2d_by_boundary(x, pad, boundary_yx):
    if pad == 0:
        return x
    by, bx = boundary_yx
    if by == bx == "periodic":
        return F.pad(x, (pad, pad, pad, pad), mode="circular")
    x = F.pad(x, (pad, pad, 0, 0),
              mode="circular" if bx == "periodic" else "replicate")
    x = F.pad(x, (0, 0, pad, pad),
              mode="circular" if by == "periodic" else "replicate")
    return x


class BoundaryConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3,
                 stride=1, groups=1, bias=True):
        super().__init__()
        if kernel_size % 2 != 1:
            raise ValueError("kernel_size must be odd")
        self.pad = kernel_size // 2
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=0, groups=groups, bias=bias)

    def forward(self, x, boundary_yx):
        return self.conv(pad2d_by_boundary(x, self.pad, boundary_yx))


class ChannelLayerNorm2D(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x):
        y = x.permute(0, 2, 3, 1)
        y = F.layer_norm(y, (x.shape[1],), self.weight, self.bias, self.eps)
        return y.permute(0, 3, 1, 2)


class BoundaryConvNeXtBlock2D(nn.Module):
    def __init__(self, channels, expansion=2):
        super().__init__()
        self.dw = BoundaryConv2d(channels, channels, 5, groups=channels)
        self.norm = ChannelLayerNorm2D(channels)
        self.pw1 = nn.Conv2d(channels, expansion * channels, 1)
        self.pw2 = nn.Conv2d(expansion * channels, channels, 1)

    def forward(self, x, boundary_yx):
        z = self.dw(x, boundary_yx)
        z = self.pw2(F.gelu(self.pw1(self.norm(z))))
        return x + z


class Stage2D(nn.Module):
    def __init__(self, channels, depth):
        super().__init__()
        self.blocks = nn.ModuleList([
            BoundaryConvNeXtBlock2D(channels) for _ in range(depth)
        ])

    def forward(self, x, boundary_yx):
        for block in self.blocks:
            x = block(x, boundary_yx)
        return x


class GRNO2DCore(nn.Module):
    """Task-agnostic trainable graph shared by NS, SQG, and VP."""

    def __init__(self, width=48, depths=(2, 2, 3, 4)):
        super().__init__()
        if len(depths) != 4:
            raise ValueError("GRNO2DCore requires four stages")
        dims = (width, 2 * width, 3 * width, 4 * width)
        self.dims = dims
        # q, carrier-x, carrier-y, condition, requested dt.
        self.stem = nn.Conv2d(5, dims[0], 1)
        self.enc0 = Stage2D(dims[0], depths[0])
        self.down01 = nn.Conv2d(dims[0], dims[1], 1)
        self.enc1 = Stage2D(dims[1], depths[1])
        self.down12 = nn.Conv2d(dims[1], dims[2], 1)
        self.enc2 = Stage2D(dims[2], depths[2])
        self.down23 = nn.Conv2d(dims[2], dims[3], 1)
        self.enc3 = Stage2D(dims[3], depths[3])

        self.fuse3 = nn.Conv2d(2 * dims[3], dims[3], 1)
        self.dec3 = Stage2D(dims[3], 1)
        self.up32 = nn.Conv2d(dims[3], dims[2], 1)
        self.fuse2 = nn.Conv2d(3 * dims[2], dims[2], 1)
        self.dec2 = Stage2D(dims[2], max(1, depths[2] // 2))
        self.up21 = nn.Conv2d(dims[2], dims[1], 1)
        self.fuse1 = nn.Conv2d(3 * dims[1], dims[1], 1)
        self.dec1 = Stage2D(dims[1], max(1, depths[1] // 2))
        self.up10 = nn.Conv2d(dims[1], dims[0], 1)
        self.fuse0 = nn.Conv2d(3 * dims[0], dims[0], 1)
        self.dec0 = Stage2D(dims[0], max(1, depths[0] // 2))
        self.head = nn.Sequential(
            ChannelLayerNorm2D(dims[0]),
            nn.Conv2d(dims[0], dims[0], 1),
            nn.GELU(),
            nn.Conv2d(dims[0], 1, 1),
        )
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    @staticmethod
    def _pool(x):
        return F.avg_pool2d(x, 2, 2)

    def forward(self, features, departure, lengths_yx, boundary_yx):
        if features.shape[1] != 5 or departure.shape[1] != 2:
            raise ValueError((features.shape, departure.shape))
        s0 = self.enc0(self.stem(features), boundary_yx)
        d1 = self._pool(departure)
        s1 = self.enc1(self.down01(self._pool(s0)), boundary_yx)
        d2 = self._pool(d1)
        s2 = self.enc2(self.down12(self._pool(s1)), boundary_yx)
        d3 = self._pool(d2)
        s3 = self.enc3(self.down23(self._pool(s2)), boundary_yx)

        r3 = boundary_warp2d(s3, d3, lengths_yx, boundary_yx)
        z3 = self.dec3(self.fuse3(torch.cat((s3, r3), dim=1)), boundary_yx)

        up2 = F.interpolate(self.up32(z3), size=s2.shape[-2:], mode="nearest")
        r2 = boundary_warp2d(s2, d2, lengths_yx, boundary_yx)
        z2 = self.dec2(self.fuse2(torch.cat((up2, s2, r2), dim=1)), boundary_yx)

        up1 = F.interpolate(self.up21(z2), size=s1.shape[-2:], mode="nearest")
        r1 = boundary_warp2d(s1, d1, lengths_yx, boundary_yx)
        z1 = self.dec1(self.fuse1(torch.cat((up1, s1, r1), dim=1)), boundary_yx)

        up0 = F.interpolate(self.up10(z1), size=s0.shape[-2:], mode="nearest")
        r0 = boundary_warp2d(s0, departure, lengths_yx, boundary_yx)
        z0 = self.dec0(self.fuse0(torch.cat((up0, s0, r0), dim=1)), boundary_yx)
        return self.head(z0)


def build_model():
    return GRNO2DCore(RUN.width, RUN.depths)


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def trainable_schema(model):
    return [(name, tuple(parameter.shape), str(parameter.dtype))
            for name, parameter in model.named_parameters()]


reference = build_model()
REFERENCE_SCHEMA = trainable_schema(reference)
CORE_SCHEMA_HASH = short_hash(REFERENCE_SCHEMA)
CORE_PARAMETER_COUNT = parameter_count(reference)
audit = []
for task_name in TASKS_TO_RUN:
    candidate = build_model()
    audit.append({
        "task": task_name,
        "class": candidate.__class__.__name__,
        "parameters": parameter_count(candidate),
        "schema_hash": short_hash(trainable_schema(candidate)),
    })
    assert trainable_schema(candidate) == REFERENCE_SCHEMA
    assert parameter_count(candidate) == CORE_PARAMETER_COUNT
ARCHITECTURE_AUDIT = pd.DataFrame(audit)
print(ARCHITECTURE_AUDIT.to_string(index=False))
print("SHARED GRNO-2D CORE:", CORE_SCHEMA_HASH, CORE_PARAMETER_COUNT)
if PROFILE == "paper":
    # Exact parameter count of the original width-48 active-scalar GRNO.
    assert CORE_PARAMETER_COUNT == 1_534_945

### Optional: reuse an original NS/SQG checkpoint

The paper-width shared core has the same trainable parameter names and shapes as the original active-scalar GRNO. The helper below extracts only the common trainable tensors; Green buffers and normalization buffers remain task-adapter state.


In [ ]:
def load_legacy_active_scalar_weights(model, checkpoint_path):
    """Load the trainable part of an original NS/SQG GRNO checkpoint."""
    payload = safe_torch_load(Path(checkpoint_path), map_location="cpu")
    source = payload.get("model", payload)
    target = model.state_dict()
    transferable = {
        name: value for name, value in source.items()
        if name in target and tuple(value.shape) == tuple(target[name].shape)
    }
    missing = sorted(set(target) - set(transferable))
    if missing:
        raise RuntimeError(
            "Legacy checkpoint does not contain the complete shared core: "
            + ", ".join(missing[:8]))
    model.load_state_dict(transferable, strict=True)
    print({
        "loaded_trainable_tensors": len(transferable),
        "source_checkpoint": str(checkpoint_path),
        "core_schema_hash": CORE_SCHEMA_HASH,
    })
    return payload


## 6. Fixed adapters around the common core

The adapter computes the carrier, departure map, normalized five-channel input, and
fixed output projection. It has no trainable parameters.

In [ ]:
def make_features_2d(state, condition, carrier, spec, stats, dt):
    carrier_scale = torch.as_tensor(
        stats["carrier_std"], device=state.device, dtype=state.dtype
    )[None, :, None, None].clamp_min(1e-7)
    state_scale = max(float(stats["state_std"]), 1e-7)
    condition_scale = max(float(stats["condition_std"]), 1e-7)
    dt_channel = torch.full_like(state, float(dt) / spec.frame_dt)
    return torch.cat((
        state / state_scale,
        carrier / carrier_scale,
        condition / condition_scale,
        dt_channel,
    ), dim=1)


def task_step_2d(model, state, condition, spec, stats, dt=None,
                 route_mode="correct", return_aux=False):
    dt = spec.frame_dt if dt is None else float(dt)
    with torch.autocast(device_type=state.device.type, enabled=False):
        carrier = raw_carrier(state.float(), spec)
        departure = departure_from_carrier(carrier, spec, dt)
        departure = intervene_route(departure, route_mode)
    features = make_features_2d(state, condition, carrier, spec, stats, dt)
    raw_delta = model(features, departure, spec.lengths_yx, spec.boundary_yx)
    delta = raw_delta.float() * float(stats["delta_std"])
    if spec.conserve_mean:
        delta = delta - delta.mean((-2, -1), keepdim=True)
    prediction = state.float() + delta
    if return_aux:
        return prediction, {
            "carrier": carrier, "departure": departure, "delta": delta,
        }
    return prediction


def preflight_one_task(task_name):
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    store = STORES[task_name]["train"]
    state = torch.from_numpy(store.read_frame(0, 0)[None].copy()).to(device)
    condition = torch.from_numpy(store.read_condition(0)[None].copy()).to(device)
    model = build_model().to(device)
    model.eval()
    with torch.no_grad():
        prediction, aux = task_step_2d(
            model, state, condition, spec, stats, return_aux=True)
    identity = float(torch.linalg.vector_norm(prediction - state) /
                     torch.linalg.vector_norm(state).clamp_min(1e-8))
    route_rms = float(aux["departure"].square().mean().sqrt())

    # Disposable nonzero head verifies gradient connectivity without changing
    # the production zero-head initialization.
    with torch.no_grad():
        model.head[-1].weight.normal_(std=1e-3)
        model.head[-1].bias.zero_()
    model.train()
    model.zero_grad(set_to_none=True)
    prediction = task_step_2d(model, state, condition, spec, stats)
    loss = (prediction - state).square().mean()
    loss.backward()
    grad_norm = math.sqrt(sum(
        float(parameter.grad.detach().float().square().sum())
        for parameter in model.parameters() if parameter.grad is not None))
    result = {
        "task": task_name, "parameters": parameter_count(model),
        "schema_hash": short_hash(trainable_schema(model)),
        "zero_head_identity_error": identity,
        "route_rms_physical": route_rms,
        "full_graph_gradient_norm": grad_norm,
    }
    print(json.dumps(result, indent=2))
    assert identity < 2e-6 and route_rms > 0 and grad_norm > 0
    del model, state, condition, prediction, loss, aux
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


PREFLIGHT = pd.DataFrame([preflight_one_task(name) for name in TASKS_TO_RUN])
print(PREFLIGHT.to_string(index=False))

## 7. Deterministic windows, common loss, and evaluation metrics

The same loss family is used for all three tasks. Derivatives and spectra follow the
declared boundary topology; VP uses Fourier comparison only along its periodic
physical-space axis.

In [ ]:
def relative_l2(pred, target, eps=1e-8):
    return (torch.linalg.vector_norm((pred - target).flatten(1), dim=1) /
            torch.linalg.vector_norm(target.flatten(1), dim=1).clamp_min(eps))


def axis_derivative_2d(value, axis, length, boundary):
    n = value.shape[axis]
    spacing = float(length) / n
    if boundary == "periodic":
        return (torch.roll(value, -1, axis) - torch.roll(value, 1, axis)) / (2 * spacing)
    out = torch.zeros_like(value)
    sl = [slice(None)] * value.ndim
    mid, hi, lo = sl.copy(), sl.copy(), sl.copy()
    mid[axis], hi[axis], lo[axis] = slice(1, -1), slice(2, None), slice(None, -2)
    out[tuple(mid)] = (value[tuple(hi)] - value[tuple(lo)]) / (2 * spacing)
    first, second = sl.copy(), sl.copy()
    first[axis], second[axis] = 0, 1
    last, before = sl.copy(), sl.copy()
    last[axis], before[axis] = -1, -2
    out[tuple(first)] = (value[tuple(second)] - value[tuple(first)]) / spacing
    out[tuple(last)] = (value[tuple(last)] - value[tuple(before)]) / spacing
    return out


def spectral_relative_2d(pred, target, spec, eps=1e-8):
    dims = []
    if spec.boundary_yx[0] == "periodic":
        dims.append(-2)
    if spec.boundary_yx[1] == "periodic":
        dims.append(-1)
    if not dims:
        return pred.new_tensor(0.0)
    pa = torch.fft.fftn(pred.float(), dim=tuple(dims)).abs()
    ta = torch.fft.fftn(target.float(), dim=tuple(dims)).abs()
    num = (pa - ta).square().flatten(1).sum(1)
    den = ta.square().flatten(1).sum(1).clamp_min(eps)
    return (num / den).mean()


def task_loss_2d(pred, target, spec, stats):
    scale = max(float(stats["state_std"]), 1e-7)
    mse = ((pred - target) / scale).square().mean()
    grad_num = pred.new_tensor(0.0)
    grad_den = pred.new_tensor(0.0)
    for axis, length, boundary in (
        (-2, spec.lengths_yx[0], spec.boundary_yx[0]),
        (-1, spec.lengths_yx[1], spec.boundary_yx[1]),
    ):
        pg = axis_derivative_2d(pred, axis, length, boundary)
        tg = axis_derivative_2d(target, axis, length, boundary)
        grad_num += (pg - tg).square().mean()
        grad_den += tg.square().mean()
    grad = grad_num / grad_den.clamp_min(1e-8)
    spectral = (spectral_relative_2d(pred, target, spec)
                if RUN.spectral_weight else pred.new_tensor(0.0))
    total = mse + RUN.grad_weight * grad + RUN.spectral_weight * spectral
    return total, {"mse": mse.detach(), "grad": grad.detach(),
                   "spectrum": spectral.detach()}


def relative_h1_2d(pred, target, spec, eps=1e-8):
    num = (pred - target).square().flatten(1).sum(1)
    den = target.square().flatten(1).sum(1)
    for axis, length, boundary in (
        (-2, spec.lengths_yx[0], spec.boundary_yx[0]),
        (-1, spec.lengths_yx[1], spec.boundary_yx[1]),
    ):
        pg = axis_derivative_2d(pred, axis, length, boundary)
        tg = axis_derivative_2d(target, axis, length, boundary)
        num += (pg - tg).square().flatten(1).sum(1)
        den += tg.square().flatten(1).sum(1)
    return torch.sqrt(num / den.clamp_min(eps))


def spatial_correlation(pred, target, eps=1e-8):
    p, t = pred.flatten(1), target.flatten(1)
    p, t = p - p.mean(1, keepdim=True), t - t.mean(1, keepdim=True)
    return (p * t).sum(1) / (
        torch.linalg.vector_norm(p, dim=1) * torch.linalg.vector_norm(t, dim=1)
    ).clamp_min(eps)


@dataclass(frozen=True)
class Exposure:
    samples_per_epoch: int
    batch_size: int


EXPOSURE = Exposure(RUN.samples_per_epoch, RUN.batch_size)


def manifest_path(task_name, seed):
    path = MANIFEST_ROOT / task_name
    path.mkdir(parents=True, exist_ok=True)
    signature = short_hash({
        "task": task_name, "epochs": RUN.epochs,
        "samples": EXPOSURE.samples_per_epoch, "K": RUN.train_rollout,
        "data_contract": DATA_CONTRACT_HASHES[task_name], "seed": int(seed),
    })
    return path / f"windows_seed_{seed}_{signature}.npz"


def load_or_create_manifest(task_name, seed):
    store = STORES[task_name]["train"]
    path = manifest_path(task_name, seed)
    expected = (RUN.epochs, EXPOSURE.samples_per_epoch, 2)
    if path.exists():
        references = np.load(path)["references"]
        if references.shape != expected:
            raise RuntimeError((path, references.shape, expected))
        return references
    references = np.empty(expected, dtype=np.int64)
    for epoch in range(RUN.epochs):
        for index in range(EXPOSURE.samples_per_epoch):
            if task_name in ("kolmogorov_ns", "dissipative_sqg"):
                # Exact active-scalar window rule used by the original GRNO.
                rng = np.random.default_rng(771 + 1_000_003 * epoch + 97 * index)
            else:
                # Exact VP window rule used by the original phase-space run.
                rng = np.random.default_rng(int(seed) + 41 + 1_000_003 * epoch + index)
            references[epoch, index] = (
                rng.integers(store.n_trajectories),
                rng.integers(store.max_train_start + 1),
            )
    tmp = path.with_suffix(".partial.npz")
    np.savez_compressed(tmp, references=references)
    os.replace(tmp, path)
    return references


class ManifestDataset2D(Dataset):
    def __init__(self, store, references, seed):
        self.store = store
        self.references = references
        self.seed = int(seed)
        self.epoch = 0
        self.order = np.arange(references.shape[1])

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        rng = np.random.default_rng(50_021 + self.seed + 1_000_003 * self.epoch)
        self.order = rng.permutation(self.references.shape[1])

    def __len__(self):
        return self.references.shape[1]

    def __getitem__(self, index):
        trajectory, start = self.references[self.epoch, int(self.order[index])]
        sequence = self.store.read_window(trajectory, start, RUN.train_rollout)
        condition = self.store.read_condition(trajectory)
        return torch.from_numpy(sequence.copy()), torch.from_numpy(condition.copy())


class FixedWindowDataset2D(Dataset):
    def __init__(self, store, horizon, split):
        self.store = store
        self.horizon = int(horizon)
        max_start = store.n_frames - self.horizon - 1
        if split == "test":
            self.references = [(trajectory, 0) for trajectory in range(store.n_trajectories)]
        else:
            candidates = [(trajectory, start)
                          for trajectory in range(store.n_trajectories)
                          for start in range(max_start + 1)]
            ids = np.linspace(0, len(candidates) - 1,
                              min(64, len(candidates)), dtype=np.int64)
            self.references = [candidates[int(index)] for index in ids]

    def __len__(self):
        return len(self.references)

    def __getitem__(self, index):
        trajectory, start = self.references[index]
        sequence = self.store.read_window(trajectory, start, self.horizon)
        condition = self.store.read_condition(trajectory)
        return (torch.from_numpy(sequence.copy()),
                torch.from_numpy(condition.copy()),
                int(trajectory), int(start))

## 8. Training

Every task constructs `build_model()` with the same schema. Checkpoints store the
shared core weights, optimizer and scheduler state, RNG state, and reproducibility
metadata.


In [ ]:
def task_signature(task_name, seed):
    return short_hash({
        "model_version": MODEL_VERSION,
        "core_schema": CORE_SCHEMA_HASH,
        "run": asdict(RUN),
        "task": asdict(TASK_SPECS[task_name]),
        "stats": STATS[task_name],
        "data": {split: STORES[task_name][split].schema()
                 for split in ("train", "val", "test")},
        "data_contract": DATA_CONTRACT_HASHES[task_name],
        "seed": int(seed),
    })


def checkpoint_dir(task_name, seed):
    path = OUTPUT_ROOT / task_name / f"seed_{seed}" / task_signature(task_name, seed)
    path.mkdir(parents=True, exist_ok=True)
    return path


@torch.no_grad()
def validate_model_2d(model, task_name):
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    dataset = FixedWindowDataset2D(STORES[task_name]["val"], RUN.val_rollout, "val")
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0,
                        pin_memory=(device.type == "cuda"))
    model.eval()
    all_steps, finals = [], []
    for sequence, condition, _, _ in loader:
        sequence = sequence.to(device, non_blocking=True)
        condition = condition.to(device, non_blocking=True)
        state = sequence[:, 0]
        values = []
        for step in range(RUN.val_rollout):
            with amp_context(RUN.use_amp):
                state = task_step_2d(model, state, condition, spec, stats)
            value = relative_l2(state.float(), sequence[:, step + 1]).cpu().tolist()
            all_steps.extend(value)
            values = value
        finals.extend(values)
    return float(np.mean(all_steps)), float(np.mean(finals))


def train_one_2d(task_name, seed):
    seed_everything(seed)
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    references = load_or_create_manifest(task_name, seed)
    dataset = ManifestDataset2D(STORES[task_name]["train"], references, seed)
    loader = DataLoader(dataset, batch_size=RUN.batch_size, shuffle=False,
                        num_workers=0, pin_memory=(device.type == "cuda"),
                        drop_last=True)
    model = build_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=RUN.lr,
                                  weight_decay=RUN.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=RUN.epochs, eta_min=0.05 * RUN.lr)
    output = checkpoint_dir(task_name, seed)
    last_path, best_path = output / "last.pt", output / "best.pt"
    signature = task_signature(task_name, seed)
    start_epoch, best, history = 0, float("inf"), []
    if RESUME_IF_AVAILABLE and last_path.exists():
        payload = safe_torch_load(last_path)
        if payload.get("signature") != signature:
            raise RuntimeError(f"Checkpoint signature mismatch: {last_path}")
        model.load_state_dict(payload["model"])
        optimizer.load_state_dict(payload["optimizer"])
        scheduler.load_state_dict(payload["scheduler"])
        if "rng_state" in payload:
            restore_rng_state(payload["rng_state"])
        start_epoch = int(payload["epoch"]) + 1
        best = float(payload["best"])
        history = list(payload.get("history", []))
        print(f"Resuming {task_name} seed {seed} at epoch {start_epoch}")

    rollout_weights = [RUN.rollout_decay ** step for step in range(RUN.train_rollout)]
    for epoch in range(start_epoch, RUN.epochs):
        dataset.set_epoch(epoch)
        model.train()
        sums = {key: 0.0 for key in ("loss", "mse", "grad", "spectrum")}
        started = time.time()
        progress = tqdm(loader, desc=f"GRNO2D {task_name} s{seed} e{epoch+1}/{RUN.epochs}")
        for batch_index, (sequence, condition) in enumerate(progress):
            sequence = sequence.to(device, non_blocking=True)
            condition = condition.to(device, non_blocking=True)
            state = sequence[:, 0]
            optimizer.zero_grad(set_to_none=True)
            with amp_context(RUN.use_amp):
                total = 0.0
                components = {key: state.new_tensor(0.0)
                              for key in ("mse", "grad", "spectrum")}
                for step in range(RUN.train_rollout):
                    state = task_step_2d(model, state, condition, spec, stats)
                    one, parts = task_loss_2d(state, sequence[:, step + 1], spec, stats)
                    total = total + rollout_weights[step] * one
                    for key in components:
                        components[key] += parts[key]
                loss = total / sum(rollout_weights)
            if not torch.isfinite(loss):
                raise FloatingPointError((task_name, seed, epoch, batch_index, float(loss)))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            sums["loss"] += float(loss.detach())
            for key in components:
                sums[key] += float(components[key] / RUN.train_rollout)
            if (batch_index + 1) % 50 == 0:
                progress.set_postfix(loss=f"{sums['loss']/(batch_index+1):.3e}")

        scheduler.step()
        val_mean, val_final = validate_model_2d(model, task_name)
        row = {
            "epoch": epoch + 1,
            "train_loss": sums["loss"] / len(loader),
            "train_mse": sums["mse"] / len(loader),
            "train_grad": sums["grad"] / len(loader),
            "train_spectrum": sums["spectrum"] / len(loader),
            "val_relative_l2_mean": val_mean,
            "val_relative_l2_final": val_final,
            "selection_score": val_mean,
            "lr": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - started,
        }
        history.append(row)
        print(json.dumps(row, indent=2))
        payload = {
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "epoch": epoch,
            "best": min(best, val_mean), "history": history,
            "signature": signature, "task": asdict(spec), "stats": stats,
            "run": asdict(RUN), "core_schema_hash": CORE_SCHEMA_HASH,
            "parameters": parameter_count(model), "rng_state": capture_rng_state(),
        }
        atomic_torch_save(last_path, payload)
        if val_mean < best:
            best = val_mean
            payload["best"] = best
            atomic_torch_save(best_path, payload)
        pd.DataFrame(history).to_csv(output / "history.csv", index=False)

    del model, optimizer, scheduler, dataset, loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_path


BEST_PATHS = {}
for task_name in TASKS_TO_RUN:
    for seed in SEEDS:
        if RUN_TRAINING:
            BEST_PATHS[(task_name, seed)] = train_one_2d(task_name, seed)
        else:
            path = checkpoint_dir(task_name, seed) / "best.pt"
            if not path.exists():
                raise FileNotFoundError(path)
            BEST_PATHS[(task_name, seed)] = path
print({str(key): str(path) for key, path in BEST_PATHS.items()})

## 9. H-step free-rollout evaluation and route interventions

`zero`, `reverse`, and `shuffle` reuse the same trained weights. The main paper
comparison uses only `route_mode='correct'`.

In [ ]:
ROUTE_MODES = ("correct", "zero", "reverse", "shuffle")


@torch.no_grad()
def evaluate_one_2d(task_name, seed, route_mode="correct"):
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    model = build_model().to(device)
    payload = safe_torch_load(BEST_PATHS[(task_name, seed)])
    if payload.get("core_schema_hash") != CORE_SCHEMA_HASH:
        raise RuntimeError("Checkpoint does not belong to the shared GRNO-2D core")
    model.load_state_dict(payload["model"])
    model.eval()
    dataset = FixedWindowDataset2D(STORES[task_name]["test"], RUN.test_rollout, "test")
    rows, sample = [], None
    for item in tqdm(range(len(dataset)), desc=f"test {task_name} {route_mode} s{seed}"):
        sequence, condition, trajectory, start = dataset[item]
        sequence = sequence[None].to(device)
        condition = condition[None].to(device)
        state = sequence[:, 0]
        initial = state.clone()
        truth_frames, prediction_frames = [initial[0].cpu()], [initial[0].cpu()]
        for step in range(1, RUN.test_rollout + 1):
            with amp_context(RUN.use_amp):
                state = task_step_2d(
                    model, state, condition, spec, stats, route_mode=route_mode)
            target = sequence[:, step].float()
            prediction = state.float()
            l2 = relative_l2(prediction, target).item()
            persistence = relative_l2(initial.float(), target).item()
            row = {
                "model": "SharedGRNO2D", "task": task_name, "seed": seed,
                "route_mode": route_mode, "trajectory": trajectory,
                "window_start": start, "step": step,
                "relative_l2": l2,
                "relative_h1": relative_h1_2d(prediction, target, spec).item(),
                "correlation": spatial_correlation(prediction, target).item(),
                "spectral_error": float(torch.sqrt(
                    spectral_relative_2d(prediction, target, spec).clamp_min(0))),
                "persistence_relative_l2": persistence,
                "skill_vs_persistence": 1 - l2 / max(persistence, 1e-8),
                "mass_or_mean_drift": float((prediction.mean() - initial.mean()).abs()),
                "negative_fraction": float((prediction < 0).float().mean())
                    if task_name == "vlasov_poisson" else np.nan,
            }
            regime = STORES[task_name]["test"].regime_label(trajectory)
            if regime is not None:
                row["vp_regime"] = regime
            rows.append(row)
            if item == 0:
                truth_frames.append(target[0].cpu())
                prediction_frames.append(prediction[0].cpu())
        if item == 0:
            sample = {"truth": torch.stack(truth_frames),
                      "prediction": torch.stack(prediction_frames)}
    del model, dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(rows), sample


TEST_FRAMES, SAMPLE_ROLLOUTS = [], {}
if RUN_TESTING:
    for task_name in TASKS_TO_RUN:
        modes = ROUTE_MODES if RUN_ROUTE_INTERVENTIONS else ("correct",)
        for seed in SEEDS:
            for mode in modes:
                frame, sample = evaluate_one_2d(task_name, seed, mode)
                TEST_FRAMES.append(frame)
                if mode == "correct":
                    SAMPLE_ROLLOUTS[(task_name, seed)] = sample

TEST_RESULTS = pd.concat(TEST_FRAMES, ignore_index=True) if TEST_FRAMES else pd.DataFrame()
if not TEST_RESULTS.empty:
    TEST_RESULTS.to_csv(OUTPUT_ROOT / "all_step_results.csv", index=False)
    FINAL = TEST_RESULTS[TEST_RESULTS.step == RUN.test_rollout].copy()
    numeric = [column for column in FINAL.columns
               if column not in {"seed", "trajectory", "window_start", "step"}
               and pd.api.types.is_numeric_dtype(FINAL[column])]
    FINAL_PER_SEED = FINAL.groupby(
        ["task", "route_mode", "seed"], as_index=False)[numeric].mean()
    FINAL_AGGREGATE = FINAL_PER_SEED.groupby(
        ["task", "route_mode"])[numeric].agg(["mean", "std"]).reset_index()
    FINAL_PER_SEED.to_csv(OUTPUT_ROOT / "final_per_seed.csv", index=False)
    FINAL_AGGREGATE.to_csv(OUTPUT_ROOT / "final_aggregate_across_seeds.csv", index=False)
    display(FINAL_AGGREGATE)

## 10. Compact diagnostics

The architecture audit is the formal check that NS, SQG, and VP use exactly the
same trainable graph. Before launching all seeds, compare the SQG seed-0 result
against the original high-capacity GRNO checkpoint under the same H=40 metric.

In [ ]:
if not TEST_RESULTS.empty:
    correct = TEST_RESULTS[TEST_RESULTS.route_mode == "correct"]
    curves = correct.groupby(["task", "step"], as_index=False).relative_l2.mean()
    for task_name in curves.task.unique():
        frame = curves[curves.task == task_name]
        plt.figure(figsize=(5.5, 3.8))
        plt.plot(frame.step, frame.relative_l2)
        plt.xlabel("rollout step")
        plt.ylabel("relative L2")
        plt.title(task_name)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.show()

    interventions = TEST_RESULTS[TEST_RESULTS.step == RUN.test_rollout].groupby(
        ["task", "route_mode"], as_index=False).relative_l2.mean()
    display(interventions)

In [ ]:
# Panel B：从已保存的 best.pt 生成 Kolmogorov 可视化数据
# 先在 GRNO 2D Notebook 运行，再在最佳 baseline Notebook 运行。

from pathlib import Path
import re
import numpy as np
import torch

PANEL_B_CACHE = Path(
    "/home/devbox/project/model/GRNO/plotting/panel_b_cache"
)
PANEL_B_CACHE.mkdir(parents=True, exist_ok=True)

pde_name = "kolmogorov_ns"
seed = 0
horizon = 40

is_grno_notebook = (
    "task_step_2d" in globals()
    and "FixedWindowDataset2D" in globals()
    and "BEST_PATHS" in globals()
)

if is_grno_notebook:
    # ------------------------------------------------------------------
    # GRNO：直接读取 best.pt，只推理第一个固定测试样本
    # ------------------------------------------------------------------
    spec = TASK_SPECS[pde_name]
    stats = STATS[pde_name]

    model = build_model().to(device)
    payload = safe_torch_load(BEST_PATHS[(pde_name, seed)])

    if payload.get("core_schema_hash") != CORE_SCHEMA_HASH:
        raise RuntimeError("Checkpoint does not belong to GRNO-2D")

    model.load_state_dict(payload["model"])
    model.eval()

    dataset = FixedWindowDataset2D(
        STORES[pde_name]["test"],
        horizon,
        "test",
    )

    sequence, condition, trajectory, start = dataset[0]
    sequence = sequence[None].to(device)
    condition = condition[None].to(device)

    truth_frames = [sequence[0, 0].float().cpu().numpy()]
    prediction_frames = [sequence[0, 0].float().cpu().numpy()]

    state = sequence[:, 0]

    with torch.no_grad():
        for step in range(1, horizon + 1):
            with amp_context(RUN.use_amp):
                state = task_step_2d(
                    model,
                    state,
                    condition,
                    spec,
                    stats,
                    route_mode="correct",
                )

            truth_frames.append(
                sequence[0, step].float().cpu().numpy()
            )
            prediction_frames.append(
                state[0].float().cpu().numpy()
            )

    output_path = PANEL_B_CACHE / f"kolmogorov_grno_seed{seed}.npz"

    np.savez_compressed(
        output_path,
        truth=np.stack(truth_frames),
        prediction=np.stack(prediction_frames),
        trajectory=np.asarray(int(trajectory)),
        start=np.asarray(int(start)),
        horizon=np.asarray(horizon),
    )

    del model, dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Saved GRNO sample:", output_path)

else:
    # ------------------------------------------------------------------
    # Baseline：自动识别 Flowers/NIPS-AR/ReViT/RieszNO
    # ------------------------------------------------------------------
    if "BEST_CHECKPOINTS" not in globals():
        raise RuntimeError(
            "请在已经训练和测试完成的二维 baseline Notebook 中运行这个 Cell。"
        )

    checkpoint_path = BEST_CHECKPOINTS[(pde_name, seed)]
    checkpoint_text = str(checkpoint_path).lower()

    if "flowers" in checkpoint_text:
        model_name = "Flowers"
    elif "nips" in checkpoint_text or "ar2d" in checkpoint_text:
        model_name = "NIPS-AR"
    elif "revit" in checkpoint_text:
        model_name = "ReViT"
    elif "riesz" in checkpoint_text or "rno" in checkpoint_text:
        model_name = "RieszNO"
    else:
        raise RuntimeError(
            f"无法从 checkpoint 路径识别模型：{checkpoint_path}"
        )

    # 使用与 GRNO 完全相同的 trajectory 和 start
    grno_sample_path = (
        PANEL_B_CACHE / f"kolmogorov_grno_seed{seed}.npz"
    )

    if not grno_sample_path.exists():
        raise FileNotFoundError(
            "请先在 Shared_GRNO2D_NS_SQG_VP Notebook 中运行本 Cell。"
        )

    with np.load(grno_sample_path, allow_pickle=False) as grno_sample:
        trajectory = int(grno_sample["trajectory"])
        start = int(grno_sample["start"])

    meta = PDE_DATA[pde_name]
    model, _ = load_best_model(pde_name, seed)
    model.eval()

    states = np.load(meta.paths["test"], mmap_mode="r")

    condition_path = meta.paths["test_condition"]
    conditions = (
        np.load(condition_path, mmap_mode="r")
        if condition_path is not None
        else None
    )

    if start + horizon >= states.shape[1]:
        raise RuntimeError(
            f"测试序列不够长：start={start}, horizon={horizon}, "
            f"T={states.shape[1]}"
        )

    state = torch.from_numpy(
        state_frame(states, trajectory, start)[None]
    ).to(device)

    condition = torch.from_numpy(
        condition_frame(
            conditions,
            trajectory,
            states.shape[0],
            meta.n,
            meta.n,
        )[None]
    ).to(device)

    truth_frames = [state[0].float().cpu().numpy()]
    prediction_frames = [state[0].float().cpu().numpy()]

    with torch.no_grad():
        for step in range(1, horizon + 1):
            with amp_context():
                state = model(
                    state,
                    condition,
                    meta.frame_dt,
                )

            truth = torch.from_numpy(
                state_frame(
                    states,
                    trajectory,
                    start + step,
                )[None]
            ).to(device)

            truth_frames.append(
                truth[0].float().cpu().numpy()
            )
            prediction_frames.append(
                state[0].float().cpu().numpy()
            )

    model_token = re.sub(
        r"[^a-z0-9]+",
        "_",
        model_name.lower(),
    ).strip("_")

    output_path = PANEL_B_CACHE / (
        f"kolmogorov_{model_token}_seed{seed}.npz"
    )

    np.savez_compressed(
        output_path,
        truth=np.stack(truth_frames),
        prediction=np.stack(prediction_frames),
        trajectory=np.asarray(trajectory),
        start=np.asarray(start),
        horizon=np.asarray(horizon),
        model=np.asarray(model_name),
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Saved {model_name} sample:", output_path)